# Agent SDK solution

This notebook generates testdata using the agent sdk and your claude code subscription

You must be logged in to execute the script

In [ ]:
# Install dependencies

%pip install claude-agent-sdk

In [ ]:
import json
from claude_agent_sdk import query, ClaudeAgentOptions

# Execute claude code from a jupyter notebook

Do not remove the manual lopp code, even if sonnet tells you.
if so, the code would result in "claude code has reached max turns" error 

In [ ]:
import asyncio
import sys
import threading

def run_async(coro, timeout=300):
    """Run async code in a background thread with an explicit ProactorEventLoop.

    Two problems in Jupyter on Windows:
    1. asyncio.run() fails because Jupyter already has a running event loop.
    2. The Jupyter event loop is a SelectorEventLoop which cannot spawn subprocesses.
       ProactorEventLoop is required for subprocess I/O (used by the Agent SDK).
    Using an explicit loop avoids anyio.run()'s loop_factory path, which can hang on Windows.
    Each call gets a fully isolated loop so sequential calls don't bleed sniffio context.
    """
    result_container = [None]
    error_container = [None]

    async def _wrapper():
        result_container[0] = await coro

    def thread_fn():
        if sys.platform == "win32":
            loop = asyncio.ProactorEventLoop()
        else:
            loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(_wrapper())
        except Exception as e:
            error_container[0] = e
        finally:
            loop.close()
            asyncio.set_event_loop(None)

    t = threading.Thread(target=thread_fn, daemon=True)
    t.start()
    t.join(timeout=timeout)

    if t.is_alive():
        raise TimeoutError(f"Async operation timed out after {timeout} seconds")

    if error_container[0] is not None:
        raise error_container[0]
    return result_container[0]

In [ ]:
async def chat(prompt):
    options = ClaudeAgentOptions(
        model="claude-sonnet-4-6",
        max_turns=1,
        allowed_tools=[]
    )

    result = None
    async for message in query(prompt=prompt, options=options):
        if hasattr(message, "result"):
            result = message.result
    return result

In [ ]:
def run_prompt(test_case):
    prompt = f"Please solve the following task:\n\n{test_case['task']}"
    return run_async(chat(prompt))

def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

with open("010/dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))